# OCDP + Snowflake Iceberg Readiness POC - Environment Setup

**Release:** v1.0.0 | **Date:** 2026-03-17

## Success Criteria Reference
| ID | Criteria | Target |
|----|----------|--------|
| SC-01 | Performance Parity | Read/write latency within <10-15% of native tables |
| SC-02 | Feature Compatibility | 100% pass rate for GA features |
| SC-03 | Governance Integrity | Horizon policies enforced with zero leakage |
| SC-04 | Resilience (HA/DR) | Successful regional failover meeting RTO/RPO |
| SC-05 | Cost Predictability | Documented TCO comparison |

## Notebook Series
| # | Notebook | Purpose |
|---|----------|--------|
| 00 | Setup Environment | Infrastructure & baseline data |
| 01 | Iceberg V3 Basics | VARIANT, nanosecond timestamps |
| 02 | Performance Benchmarks | Native vs Iceberg comparison |
| 03 | DML Operations | INSERT/UPDATE/DELETE/MERGE, time travel |
| 04 | Streams/Tasks/DT | CDC, Dynamic Tables |
| 05 | Governance | Masking, RAP, Tags |
| 06 | HA/DR | Replication templates |
| 07 | Concurrency | Stress testing |
| 08 | Databricks Interop | CLD + Unity Catalog |
| 09 | Results Summary | POC validation |

In [1]:
-- Set context
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE COMPUTE_WH;

SyntaxError: invalid syntax (1368534200.py, line 1)

---
## Step 1: Create POC Database and Schemas

In [ ]:
-- Create POC Database
CREATE DATABASE IF NOT EXISTS ICEBERG_POC;
USE DATABASE ICEBERG_POC;

-- Create schemas
CREATE SCHEMA IF NOT EXISTS TESTS;           -- Iceberg tables with customer-managed storage (External Volume)
CREATE SCHEMA IF NOT EXISTS MANAGED_ICEBERG; -- Iceberg tables with Snowflake-managed storage (no External Volume)
CREATE SCHEMA IF NOT EXISTS NATIVE_BASELINE; -- Native tables for comparison
CREATE SCHEMA IF NOT EXISTS EXTERNAL_ICEBERG;-- Iceberg tables with explicit BASE_LOCATION for interop
CREATE SCHEMA IF NOT EXISTS NOTEBOOKS;       -- For notebook deployment

SHOW SCHEMAS IN DATABASE ICEBERG_POC;

---
## Step 2: Verify External Volume

In [ ]:
-- Check External Volume exists
SHOW EXTERNAL VOLUMES LIKE 'EXVOL';

In [ ]:
-- Describe External Volume configuration
DESCRIBE EXTERNAL VOLUME EXVOL;

---
## Step 3: Create Warehouses for Benchmarking

In [ ]:
-- Create warehouses of different sizes for performance testing
CREATE WAREHOUSE IF NOT EXISTS POC_WH_XS 
    WITH WAREHOUSE_SIZE = 'XSMALL' 
    AUTO_SUSPEND = 60 
    AUTO_RESUME = TRUE;

CREATE WAREHOUSE IF NOT EXISTS POC_WH_M 
    WITH WAREHOUSE_SIZE = 'MEDIUM' 
    AUTO_SUSPEND = 60 
    AUTO_RESUME = TRUE;

CREATE WAREHOUSE IF NOT EXISTS POC_WH_L 
    WITH WAREHOUSE_SIZE = 'LARGE' 
    AUTO_SUSPEND = 60 
    AUTO_RESUME = TRUE;

SHOW WAREHOUSES LIKE 'POC_WH%';

---
## Step 4: Create Baseline Datasets - EVENTS Table
Create identical datasets in both Native and Iceberg formats for comparison testing.

In [ ]:
-- Create Native baseline healthcare tables (PATIENTS, ENCOUNTERS, CLAIMS, MEDICATIONS, PROVIDERS)
CREATE OR REPLACE TABLE ICEBERG_POC.NATIVE_BASELINE.PATIENTS (
    patient_id      INT,
    first_name      VARCHAR,
    last_name       VARCHAR,
    date_of_birth   DATE,
    gender          VARCHAR,
    blood_type      VARCHAR,
    primary_phone   VARCHAR,
    city            VARCHAR,
    state           VARCHAR,
    insurance_plan  VARCHAR
);

CREATE OR REPLACE TABLE ICEBERG_POC.NATIVE_BASELINE.ENCOUNTERS (
    encounter_id            INT,
    patient_id              INT,
    provider_id             INT,
    encounter_date          DATE,
    encounter_type          VARCHAR,
    primary_diagnosis_code  VARCHAR,
    primary_diagnosis_desc  VARCHAR,
    disposition             VARCHAR,
    total_charge            DECIMAL(10,2)
);

CREATE OR REPLACE TABLE ICEBERG_POC.NATIVE_BASELINE.CLAIMS (
    claim_id        INT,
    encounter_id    INT,
    patient_id      INT,
    payer_name      VARCHAR,
    claim_status    VARCHAR,
    submitted_date  DATE,
    paid_date       DATE,
    billed_amount   DECIMAL(10,2),
    allowed_amount  DECIMAL(10,2),
    paid_amount     DECIMAL(10,2)
);

CREATE OR REPLACE TABLE ICEBERG_POC.NATIVE_BASELINE.MEDICATIONS (
    medication_id           INT,
    patient_id              INT,
    encounter_id            INT,
    drug_name               VARCHAR,
    ndc_code                VARCHAR,
    dosage                  VARCHAR,
    frequency               VARCHAR,
    prescribed_date         DATE,
    end_date                DATE,
    prescribing_provider_id INT
);

CREATE OR REPLACE TABLE ICEBERG_POC.NATIVE_BASELINE.PROVIDERS (
    provider_id         INT,
    first_name          VARCHAR,
    last_name           VARCHAR,
    specialty           VARCHAR,
    npi_number          VARCHAR,
    facility_name       VARCHAR,
    facility_city       VARCHAR,
    facility_state      VARCHAR,
    accepting_patients  BOOLEAN
);

In [ ]:
-- Create Iceberg v3 healthcare tables (PATIENTS, ENCOUNTERS, CLAIMS, MEDICATIONS, PROVIDERS)
CREATE OR REPLACE ICEBERG TABLE ICEBERG_POC.TESTS.PATIENTS (
    patient_id      INT,
    first_name      VARCHAR,
    last_name       VARCHAR,
    date_of_birth   DATE,
    gender          VARCHAR,
    blood_type      VARCHAR,
    primary_phone   VARCHAR,
    city            VARCHAR,
    state           VARCHAR,
    insurance_plan  VARCHAR
)
CATALOG = SNOWFLAKE
EXTERNAL_VOLUME = EXVOL
ICEBERG_VERSION = 3;

CREATE OR REPLACE ICEBERG TABLE ICEBERG_POC.TESTS.ENCOUNTERS (
    encounter_id            INT,
    patient_id              INT,
    provider_id             INT,
    encounter_date          DATE,
    encounter_type          VARCHAR,
    primary_diagnosis_code  VARCHAR,
    primary_diagnosis_desc  VARCHAR,
    disposition             VARCHAR,
    total_charge            DECIMAL(10,2)
)
CATALOG = SNOWFLAKE
EXTERNAL_VOLUME = EXVOL
ICEBERG_VERSION = 3;

CREATE OR REPLACE ICEBERG TABLE ICEBERG_POC.TESTS.CLAIMS (
    claim_id        INT,
    encounter_id    INT,
    patient_id      INT,
    payer_name      VARCHAR,
    claim_status    VARCHAR,
    submitted_date  DATE,
    paid_date       DATE,
    billed_amount   DECIMAL(10,2),
    allowed_amount  DECIMAL(10,2),
    paid_amount     DECIMAL(10,2)
)
CATALOG = SNOWFLAKE
EXTERNAL_VOLUME = EXVOL
ICEBERG_VERSION = 3;

CREATE OR REPLACE ICEBERG TABLE ICEBERG_POC.TESTS.MEDICATIONS (
    medication_id           INT,
    patient_id              INT,
    encounter_id            INT,
    drug_name               VARCHAR,
    ndc_code                VARCHAR,
    dosage                  VARCHAR,
    frequency               VARCHAR,
    prescribed_date         DATE,
    end_date                DATE,
    prescribing_provider_id INT
)
CATALOG = SNOWFLAKE
EXTERNAL_VOLUME = EXVOL
ICEBERG_VERSION = 3;

CREATE OR REPLACE ICEBERG TABLE ICEBERG_POC.TESTS.PROVIDERS (
    provider_id         INT,
    first_name          VARCHAR,
    last_name           VARCHAR,
    specialty           VARCHAR,
    npi_number          VARCHAR,
    facility_name       VARCHAR,
    facility_city       VARCHAR,
    facility_state      VARCHAR,
    accepting_patients  BOOLEAN
)
CATALOG = SNOWFLAKE
EXTERNAL_VOLUME = EXVOL
ICEBERG_VERSION = 3;

In [ ]:
-- Create Iceberg v3 tables with MANAGED STORAGE (no External Volume required)
-- Syntax: CATALOG = 'SNOWFLAKE' + ICEBERG_VERSION = 3 (no EXTERNAL_VOLUME)
CREATE OR REPLACE ICEBERG TABLE ICEBERG_POC.MANAGED_ICEBERG.PATIENTS (
    patient_id      INT,
    first_name      VARCHAR,
    last_name       VARCHAR,
    date_of_birth   DATE,
    gender          VARCHAR,
    blood_type      VARCHAR,
    primary_phone   VARCHAR,
    city            VARCHAR,
    state           VARCHAR,
    insurance_plan  VARCHAR
)
CATALOG = 'SNOWFLAKE'
ICEBERG_VERSION = 3;

CREATE OR REPLACE ICEBERG TABLE ICEBERG_POC.MANAGED_ICEBERG.ENCOUNTERS (
    encounter_id            INT,
    patient_id              INT,
    provider_id             INT,
    encounter_date          DATE,
    encounter_type          VARCHAR,
    primary_diagnosis_code  VARCHAR,
    primary_diagnosis_desc  VARCHAR,
    disposition             VARCHAR,
    total_charge            DECIMAL(10,2)
)
CATALOG = 'SNOWFLAKE'
ICEBERG_VERSION = 3;

CREATE OR REPLACE ICEBERG TABLE ICEBERG_POC.MANAGED_ICEBERG.CLAIMS (
    claim_id        INT,
    encounter_id    INT,
    patient_id      INT,
    payer_name      VARCHAR,
    claim_status    VARCHAR,
    submitted_date  DATE,
    paid_date       DATE,
    billed_amount   DECIMAL(10,2),
    allowed_amount  DECIMAL(10,2),
    paid_amount     DECIMAL(10,2)
)
CATALOG = 'SNOWFLAKE'
ICEBERG_VERSION = 3;

CREATE OR REPLACE ICEBERG TABLE ICEBERG_POC.MANAGED_ICEBERG.MEDICATIONS (
    medication_id           INT,
    patient_id              INT,
    encounter_id            INT,
    drug_name               VARCHAR,
    ndc_code                VARCHAR,
    dosage                  VARCHAR,
    frequency               VARCHAR,
    prescribed_date         DATE,
    end_date                DATE,
    prescribing_provider_id INT
)
CATALOG = 'SNOWFLAKE'
ICEBERG_VERSION = 3;

CREATE OR REPLACE ICEBERG TABLE ICEBERG_POC.MANAGED_ICEBERG.PROVIDERS (
    provider_id         INT,
    first_name          VARCHAR,
    last_name           VARCHAR,
    specialty           VARCHAR,
    npi_number          VARCHAR,
    facility_name       VARCHAR,
    facility_city       VARCHAR,
    facility_state      VARCHAR,
    accepting_patients  BOOLEAN
)
CATALOG = 'SNOWFLAKE'
ICEBERG_VERSION = 3;

In [ ]:
-- Generate 1M rows of ENCOUNTERS data for Native baseline
INSERT INTO ICEBERG_POC.NATIVE_BASELINE.ENCOUNTERS
SELECT
    SEQ4() + 3001                                                                                          AS encounter_id,
    SEQ4() % 100000 + 1001                                                                                AS patient_id,
    SEQ4() % 1000 + 2001                                                                                  AS provider_id,
    DATEADD(day, -(SEQ4() % 730), CURRENT_DATE())                                                        AS encounter_date,
    ARRAY_CONSTRUCT('Outpatient Visit','Emergency','Inpatient','Specialist Referral','Preventive Care')[SEQ4() % 5]::VARCHAR AS encounter_type,
    ARRAY_CONSTRUCT('E11.9','I10','J06.9','M54.5','Z00.00','F32.1','K21.0','N39.0')[SEQ4() % 8]::VARCHAR  AS primary_diagnosis_code,
    ARRAY_CONSTRUCT('Type 2 diabetes','Hypertension','Upper respiratory infection','Low back pain','Annual wellness visit','Depression','GERD','Urinary tract infection')[SEQ4() % 8]::VARCHAR AS primary_diagnosis_desc,
    ARRAY_CONSTRUCT('Discharged','Admitted','Transferred','Observation')[SEQ4() % 4]::VARCHAR              AS disposition,
    ROUND((SEQ4() % 9900 + 100) / 10.0, 2)                                                               AS total_charge
FROM TABLE(GENERATOR(ROWCOUNT => 1000000));

-- Generate 100K PATIENTS for Native baseline
INSERT INTO ICEBERG_POC.NATIVE_BASELINE.PATIENTS
SELECT
    SEQ4() + 1001                                                                                                               AS patient_id,
    ARRAY_CONSTRUCT('Maria','James','Priya','Robert','Aisha','David','Elena','Michael','Sarah','Carlos')[SEQ4() % 10]::VARCHAR   AS first_name,
    ARRAY_CONSTRUCT('Santos','OBrien','Patel','Chen','Johnson','Kim','Rodriguez','Thompson','Williams','Nguyen')[SEQ4() % 10]::VARCHAR AS last_name,
    DATEADD(day, -(SEQ4() % 36500 + 6570), '2000-01-01'::DATE)                                                                 AS date_of_birth,
    CASE SEQ4() % 2 WHEN 0 THEN 'M' ELSE 'F' END                                                                               AS gender,
    ARRAY_CONSTRUCT('O+','A-','B+','O+','O-','AB+','A+','B-')[SEQ4() % 8]::VARCHAR                                             AS blood_type,
    '555-' || LPAD((SEQ4() % 9000 + 1000)::VARCHAR, 4, '0')                                                                    AS primary_phone,
    ARRAY_CONSTRUCT('Phoenix','Denver','Seattle','Austin','Chicago','Portland','Miami','Boston','Dallas','Atlanta')[SEQ4() % 10]::VARCHAR AS city,
    ARRAY_CONSTRUCT('AZ','CO','WA','TX','IL','OR','FL','MA','TX','GA')[SEQ4() % 10]::VARCHAR                                    AS state,
    ARRAY_CONSTRUCT('Blue Cross PPO','Aetna HMO','United Healthcare','Cigna EPO','Humana Gold','Medicare Advantage')[SEQ4() % 6]::VARCHAR AS insurance_plan
FROM TABLE(GENERATOR(ROWCOUNT => 100000));

-- Generate 500K CLAIMS for Native baseline
INSERT INTO ICEBERG_POC.NATIVE_BASELINE.CLAIMS
SELECT
    SEQ4() + 4001                                                                                         AS claim_id,
    SEQ4() % 1000000 + 3001                                                                              AS encounter_id,
    SEQ4() % 100000 + 1001                                                                               AS patient_id,
    ARRAY_CONSTRUCT('Blue Cross','Aetna','United Healthcare','Cigna','Humana','Medicare')[SEQ4() % 6]::VARCHAR AS payer_name,
    ARRAY_CONSTRUCT('Paid','Submitted','In Review','Denied')[SEQ4() % 4]::VARCHAR                        AS claim_status,
    DATEADD(day, -(SEQ4() % 365), CURRENT_DATE())                                                       AS submitted_date,
    CASE WHEN SEQ4() % 4 = 0 THEN NULL ELSE DATEADD(day, (SEQ4() % 45 + 15), DATEADD(day, -(SEQ4() % 365), CURRENT_DATE())) END AS paid_date,
    ROUND((SEQ4() % 9900 + 100) / 10.0, 2)                                                              AS billed_amount,
    ROUND((SEQ4() % 9900 + 100) / 10.0 * 0.85, 2)                                                      AS allowed_amount,
    CASE WHEN SEQ4() % 4 = 0 THEN NULL ELSE ROUND((SEQ4() % 9900 + 100) / 10.0 * 0.70, 2) END          AS paid_amount
FROM TABLE(GENERATOR(ROWCOUNT => 500000));

-- Generate 300K MEDICATIONS for Native baseline
INSERT INTO ICEBERG_POC.NATIVE_BASELINE.MEDICATIONS
SELECT
    SEQ4() + 5001                                                                                         AS medication_id,
    SEQ4() % 100000 + 1001                                                                               AS patient_id,
    SEQ4() % 1000000 + 3001                                                                              AS encounter_id,
    ARRAY_CONSTRUCT('Metformin','Lisinopril','Atorvastatin','Amoxicillin','Amlodipine','Omeprazole','Sertraline','Albuterol')[SEQ4() % 8]::VARCHAR AS drug_name,
    LPAD((SEQ4() % 90000 + 10000)::VARCHAR, 5, '0') || '-' || LPAD((SEQ4() % 9000 + 1000)::VARCHAR, 4, '0') || '-' || LPAD((SEQ4() % 90 + 10)::VARCHAR, 2, '0') AS ndc_code,
    ARRAY_CONSTRUCT('500mg','10mg','20mg','40mg','250mg','5mg','100mg','90mcg')[SEQ4() % 8]::VARCHAR      AS dosage,
    ARRAY_CONSTRUCT('Once daily','Twice daily','Three times daily','As needed','Weekly')[SEQ4() % 5]::VARCHAR AS frequency,
    DATEADD(day, -(SEQ4() % 365), CURRENT_DATE())                                                       AS prescribed_date,
    DATEADD(day, (SEQ4() % 365 + 30), DATEADD(day, -(SEQ4() % 365), CURRENT_DATE()))                  AS end_date,
    SEQ4() % 1000 + 2001                                                                                 AS prescribing_provider_id
FROM TABLE(GENERATOR(ROWCOUNT => 300000));

-- Generate 1K PROVIDERS for Native baseline
INSERT INTO ICEBERG_POC.NATIVE_BASELINE.PROVIDERS
SELECT
    SEQ4() + 2001                                                                                                               AS provider_id,
    ARRAY_CONSTRUCT('Sarah','John','Lisa','Ahmed','Rachel','Michael','Jessica','David','Susan','Robert')[SEQ4() % 10]::VARCHAR   AS first_name,
    ARRAY_CONSTRUCT('Williams','Martinez','Chang','Hassan','Green','Thompson','Lee','Kim','Brown','Davis')[SEQ4() % 10]::VARCHAR AS last_name,
    ARRAY_CONSTRUCT('Cardiology','Internal Medicine','Endocrinology','Orthopedics','Family Medicine','Oncology','Neurology','Radiology')[SEQ4() % 8]::VARCHAR AS specialty,
    LPAD((SEQ4() % 9000000000 + 1000000000)::VARCHAR, 10, '0')                                                                 AS npi_number,
    ARRAY_CONSTRUCT('Phoenix Heart Center','Denver Medical Group','Northwest Diabetes Clinic','Texas Bone and Joint','Midwest Family Health','Pacific Cancer Center')[SEQ4() % 6]::VARCHAR AS facility_name,
    ARRAY_CONSTRUCT('Phoenix','Denver','Seattle','Austin','Chicago','Portland')[SEQ4() % 6]::VARCHAR                            AS facility_city,
    ARRAY_CONSTRUCT('AZ','CO','WA','TX','IL','OR')[SEQ4() % 6]::VARCHAR                                                        AS facility_state,
    CASE SEQ4() % 5 WHEN 0 THEN FALSE ELSE TRUE END                                                                            AS accepting_patients
FROM TABLE(GENERATOR(ROWCOUNT => 1000));

In [ ]:
-- Copy all healthcare data to Iceberg tables
INSERT INTO ICEBERG_POC.TESTS.ENCOUNTERS   SELECT * FROM ICEBERG_POC.NATIVE_BASELINE.ENCOUNTERS;
INSERT INTO ICEBERG_POC.TESTS.PATIENTS     SELECT * FROM ICEBERG_POC.NATIVE_BASELINE.PATIENTS;
INSERT INTO ICEBERG_POC.TESTS.CLAIMS       SELECT * FROM ICEBERG_POC.NATIVE_BASELINE.CLAIMS;
INSERT INTO ICEBERG_POC.TESTS.MEDICATIONS  SELECT * FROM ICEBERG_POC.NATIVE_BASELINE.MEDICATIONS;
INSERT INTO ICEBERG_POC.TESTS.PROVIDERS    SELECT * FROM ICEBERG_POC.NATIVE_BASELINE.PROVIDERS;

In [ ]:
-- Copy all healthcare data to Managed Storage Iceberg tables
INSERT INTO ICEBERG_POC.MANAGED_ICEBERG.ENCOUNTERS   SELECT * FROM ICEBERG_POC.NATIVE_BASELINE.ENCOUNTERS;
INSERT INTO ICEBERG_POC.MANAGED_ICEBERG.PATIENTS     SELECT * FROM ICEBERG_POC.NATIVE_BASELINE.PATIENTS;
INSERT INTO ICEBERG_POC.MANAGED_ICEBERG.CLAIMS       SELECT * FROM ICEBERG_POC.NATIVE_BASELINE.CLAIMS;
INSERT INTO ICEBERG_POC.MANAGED_ICEBERG.MEDICATIONS  SELECT * FROM ICEBERG_POC.NATIVE_BASELINE.MEDICATIONS;
INSERT INTO ICEBERG_POC.MANAGED_ICEBERG.PROVIDERS    SELECT * FROM ICEBERG_POC.NATIVE_BASELINE.PROVIDERS;

---
## Step 5: Create Additional Baseline Tables
Create matching Native and Iceberg tables for DML, CDC, Governance, HA/DR, and Concurrency tests.

In [ ]:
-- Create External Iceberg tables (explicit BASE_LOCATION for Spark/Databricks interop)
CREATE OR REPLACE ICEBERG TABLE ICEBERG_POC.EXTERNAL_ICEBERG.PATIENTS (
    patient_id      INT,
    first_name      VARCHAR,
    last_name       VARCHAR,
    date_of_birth   DATE,
    gender          VARCHAR,
    blood_type      VARCHAR,
    primary_phone   VARCHAR,
    city            VARCHAR,
    state           VARCHAR,
    insurance_plan  VARCHAR
)
CATALOG = SNOWFLAKE
EXTERNAL_VOLUME = EXVOL
BASE_LOCATION = 'external_iceberg/patients.WlOUec7k/'
ICEBERG_VERSION = 3;

CREATE OR REPLACE ICEBERG TABLE ICEBERG_POC.EXTERNAL_ICEBERG.ENCOUNTERS (
    encounter_id            INT,
    patient_id              INT,
    provider_id             INT,
    encounter_date          DATE,
    encounter_type          VARCHAR,
    primary_diagnosis_code  VARCHAR,
    primary_diagnosis_desc  VARCHAR,
    disposition             VARCHAR,
    total_charge            DECIMAL(10,2)
)
CATALOG = SNOWFLAKE
EXTERNAL_VOLUME = EXVOL
BASE_LOCATION = 'external_iceberg/encounters.NbnupqZC/'
ICEBERG_VERSION = 3;

CREATE OR REPLACE ICEBERG TABLE ICEBERG_POC.EXTERNAL_ICEBERG.CLAIMS (
    claim_id        INT,
    encounter_id    INT,
    patient_id      INT,
    payer_name      VARCHAR,
    claim_status    VARCHAR,
    submitted_date  DATE,
    paid_date       DATE,
    billed_amount   DECIMAL(10,2),
    allowed_amount  DECIMAL(10,2),
    paid_amount     DECIMAL(10,2)
)
CATALOG = SNOWFLAKE
EXTERNAL_VOLUME = EXVOL
BASE_LOCATION = 'external_iceberg/claims.DDRle8p8/'
ICEBERG_VERSION = 3;

CREATE OR REPLACE ICEBERG TABLE ICEBERG_POC.EXTERNAL_ICEBERG.MEDICATIONS (
    medication_id           INT,
    patient_id              INT,
    encounter_id            INT,
    drug_name               VARCHAR,
    ndc_code                VARCHAR,
    dosage                  VARCHAR,
    frequency               VARCHAR,
    prescribed_date         DATE,
    end_date                DATE,
    prescribing_provider_id INT,
    medication_details      VARIANT
)
CATALOG = SNOWFLAKE
EXTERNAL_VOLUME = EXVOL
BASE_LOCATION = 'external_iceberg/medications.e98xGtj1/'
ICEBERG_VERSION = 3;

CREATE OR REPLACE ICEBERG TABLE ICEBERG_POC.EXTERNAL_ICEBERG.PROVIDERS (
    provider_id         INT,
    first_name          VARCHAR,
    last_name           VARCHAR,
    specialty           VARCHAR,
    npi_number          VARCHAR,
    facility_name       VARCHAR,
    facility_city       VARCHAR,
    facility_state      VARCHAR,
    accepting_patients  BOOLEAN,
    provider_attributes VARIANT
)
CATALOG = SNOWFLAKE
EXTERNAL_VOLUME = EXVOL
BASE_LOCATION = 'external_iceberg/providers.LnJFXcFk/'
ICEBERG_VERSION = 3;

In [ ]:
-- Seed External Iceberg tables from NATIVE_BASELINE (not TESTS to avoid dependency issues)
INSERT INTO ICEBERG_POC.EXTERNAL_ICEBERG.PATIENTS
SELECT * FROM ICEBERG_POC.NATIVE_BASELINE.PATIENTS;

INSERT INTO ICEBERG_POC.EXTERNAL_ICEBERG.ENCOUNTERS
SELECT * FROM ICEBERG_POC.NATIVE_BASELINE.ENCOUNTERS;

INSERT INTO ICEBERG_POC.EXTERNAL_ICEBERG.CLAIMS
SELECT * FROM ICEBERG_POC.NATIVE_BASELINE.CLAIMS;

INSERT INTO ICEBERG_POC.EXTERNAL_ICEBERG.MEDICATIONS
SELECT
    medication_id, patient_id, encounter_id, drug_name, ndc_code, dosage, frequency,
    prescribed_date, end_date, prescribing_provider_id,
    OBJECT_CONSTRUCT(
        'form',        ARRAY_CONSTRUCT('tablet','capsule','liquid','injection','inhaler')[medication_id % 5]::VARCHAR,
        'controlled',  CASE medication_id % 10 WHEN 0 THEN TRUE ELSE FALSE END,
        'refills',     (medication_id % 6)
    ) AS medication_details
FROM ICEBERG_POC.NATIVE_BASELINE.MEDICATIONS;

INSERT INTO ICEBERG_POC.EXTERNAL_ICEBERG.PROVIDERS
SELECT
    provider_id, first_name, last_name, specialty, npi_number,
    facility_name, facility_city, facility_state, accepting_patients,
    OBJECT_CONSTRUCT(
        'board_certified', TRUE,
        'languages',       ARRAY_CONSTRUCT('English','Spanish'),
        'telehealth',      CASE provider_id % 2 WHEN 0 THEN TRUE ELSE FALSE END
    ) AS provider_attributes
FROM ICEBERG_POC.NATIVE_BASELINE.PROVIDERS;

---
## Step 6: Verify All Tables Created

In [ ]:
-- Verify row counts match for all tables (4 storage types)
SELECT table_name, storage_type, row_count FROM (
    SELECT 'PATIENTS'    AS table_name, '1-Native'   AS storage_type, COUNT(*) AS row_count FROM ICEBERG_POC.NATIVE_BASELINE.PATIENTS
    UNION ALL SELECT 'PATIENTS',   '2-Managed',  COUNT(*) FROM ICEBERG_POC.MANAGED_ICEBERG.PATIENTS
    UNION ALL SELECT 'PATIENTS',   '3-Customer', COUNT(*) FROM ICEBERG_POC.TESTS.PATIENTS
    UNION ALL SELECT 'PATIENTS',   '4-External', COUNT(*) FROM ICEBERG_POC.EXTERNAL_ICEBERG.PATIENTS
    UNION ALL SELECT 'ENCOUNTERS', '1-Native',   COUNT(*) FROM ICEBERG_POC.NATIVE_BASELINE.ENCOUNTERS
    UNION ALL SELECT 'ENCOUNTERS', '2-Managed',  COUNT(*) FROM ICEBERG_POC.MANAGED_ICEBERG.ENCOUNTERS
    UNION ALL SELECT 'ENCOUNTERS', '3-Customer', COUNT(*) FROM ICEBERG_POC.TESTS.ENCOUNTERS
    UNION ALL SELECT 'ENCOUNTERS', '4-External', COUNT(*) FROM ICEBERG_POC.EXTERNAL_ICEBERG.ENCOUNTERS
    UNION ALL SELECT 'CLAIMS',     '1-Native',   COUNT(*) FROM ICEBERG_POC.NATIVE_BASELINE.CLAIMS
    UNION ALL SELECT 'CLAIMS',     '2-Managed',  COUNT(*) FROM ICEBERG_POC.MANAGED_ICEBERG.CLAIMS
    UNION ALL SELECT 'CLAIMS',     '3-Customer', COUNT(*) FROM ICEBERG_POC.TESTS.CLAIMS
    UNION ALL SELECT 'CLAIMS',     '4-External', COUNT(*) FROM ICEBERG_POC.EXTERNAL_ICEBERG.CLAIMS
    UNION ALL SELECT 'MEDICATIONS','1-Native',   COUNT(*) FROM ICEBERG_POC.NATIVE_BASELINE.MEDICATIONS
    UNION ALL SELECT 'MEDICATIONS','2-Managed',  COUNT(*) FROM ICEBERG_POC.MANAGED_ICEBERG.MEDICATIONS
    UNION ALL SELECT 'MEDICATIONS','3-Customer', COUNT(*) FROM ICEBERG_POC.TESTS.MEDICATIONS
    UNION ALL SELECT 'MEDICATIONS','4-External', COUNT(*) FROM ICEBERG_POC.EXTERNAL_ICEBERG.MEDICATIONS
    UNION ALL SELECT 'PROVIDERS',  '1-Native',   COUNT(*) FROM ICEBERG_POC.NATIVE_BASELINE.PROVIDERS
    UNION ALL SELECT 'PROVIDERS',  '2-Managed',  COUNT(*) FROM ICEBERG_POC.MANAGED_ICEBERG.PROVIDERS
    UNION ALL SELECT 'PROVIDERS',  '3-Customer', COUNT(*) FROM ICEBERG_POC.TESTS.PROVIDERS
    UNION ALL SELECT 'PROVIDERS',  '4-External', COUNT(*) FROM ICEBERG_POC.EXTERNAL_ICEBERG.PROVIDERS
) ORDER BY table_name, storage_type;

In [ ]:
-- Show all Native baseline tables
SHOW TABLES IN SCHEMA ICEBERG_POC.NATIVE_BASELINE;

In [ ]:
-- Show all Iceberg tables
SHOW ICEBERG TABLES IN SCHEMA ICEBERG_POC.TESTS;

---
## Step 7: Verify Iceberg Table Metadata

In [ ]:
-- Get Iceberg metadata information for ENCOUNTERS table
SELECT SYSTEM$GET_ICEBERG_TABLE_INFORMATION('ICEBERG_POC.TESTS.ENCOUNTERS') AS iceberg_info;

---
## Step 8: Sample Data Validation

In [ ]:
-- Sample from Native ENCOUNTERS table
SELECT * FROM ICEBERG_POC.NATIVE_BASELINE.ENCOUNTERS LIMIT 5;

In [ ]:
-- Sample from Iceberg ENCOUNTERS table
SELECT * FROM ICEBERG_POC.TESTS.ENCOUNTERS LIMIT 5;

In [ ]:
-- Data distribution: encounters by type, state, and avg charge
SELECT
    e.encounter_type,
    p.state,
    COUNT(*)             AS encounter_count,
    ROUND(AVG(e.total_charge), 2) AS avg_charge
FROM ICEBERG_POC.TESTS.ENCOUNTERS e
JOIN ICEBERG_POC.TESTS.PATIENTS   p ON e.patient_id = p.patient_id
GROUP BY 1, 2
ORDER BY 1, 2;

---
## Step 9: Databricks Integration Status

In [1]:
-- Check catalog integration for Databricks
SHOW CATALOG INTEGRATIONS LIKE 'gf_interop%';

SyntaxError: invalid syntax (481398726.py, line 1)

In [ ]:
-- Check CLD database
SHOW DATABASES LIKE 'gf_dbx_cld';

In [ ]:
-- List CLD tables (from Databricks Unity Catalog)
SHOW ICEBERG TABLES IN SCHEMA gf_dbx_cld.uniform;

---
## Setup Complete

### Infrastructure Created
- **Database**: ICEBERG_POC
- **Schemas**:
  - `NATIVE_BASELINE` — Standard Snowflake native tables (baseline)
  - `MANAGED_ICEBERG` — Iceberg v3 with **Snowflake-managed storage** (no External Volume)
  - `TESTS` — Iceberg v3 with **customer-managed storage** (External Volume EXVOL)
  - `EXTERNAL_ICEBERG` — Iceberg v3 with **explicit BASE_LOCATION** (for Databricks interop)
- **Warehouses**: POC_WH_XS, POC_WH_M, POC_WH_L

### Storage Type Comparison
| Schema | Storage Type | DDL Syntax | Use Case |
|--------|--------------|------------|----------|
| NATIVE_BASELINE | Snowflake Native | Standard CREATE TABLE | Baseline comparison |
| MANAGED_ICEBERG | Snowflake Managed | `CATALOG='SNOWFLAKE' ICEBERG_VERSION=3` | Zero-config, fully managed |
| TESTS | Customer External Volume | `CATALOG=SNOWFLAKE EXTERNAL_VOLUME=EXVOL` | Customer owns storage |
| EXTERNAL_ICEBERG | Customer + Explicit Path | `+ BASE_LOCATION='path/'` | Interop with external engines |

### Healthcare Tables Created (4 storage types x 5 tables = 20 tables)
| Table | Rows | Purpose |
|-------|------|---------|
| ENCOUNTERS | 1,000,000 | Main benchmark table — visit records with ICD-10 diagnoses |
| PATIENTS | 100,000 | PHI governance, masking, row access policies |
| CLAIMS | 500,000 | DML operations — INSERT/UPDATE/DELETE/MERGE |
| MEDICATIONS | 300,000 | VARIANT showcase (medication_details), concurrency target |
| PROVIDERS | 1,000 | Reference/dimension table, provider attributes VARIANT |

### Column Alignment with gf_dbx_cld.uniform
All column names match the Databricks Unity Catalog tables exactly, enabling seamless cross-platform joins in notebook 08.

### External Dependencies
- **External Volume**: EXVOL (Azure Blob Storage) — for TESTS and EXTERNAL_ICEBERG schemas
- **CLD Database**: gf_dbx_cld (Databricks Unity Catalog — same 5 healthcare tables)

### Next Steps
Run the remaining notebooks in order:
1. `01_Iceberg_V3_Basics.ipynb` - VARIANT and nanosecond timestamps with clinical data
2. `02_Performance_Benchmarks.ipynb` - Compare Native vs Managed vs Customer storage performance
3. Continue through notebook 09...